# mesh08 — 검증 ② 실물·물리: 진짜 드론과 물리학에 얼마나 가까운가

> ⚠ **이 노트북은 생성물이다.** 수정은 `report_mesh/src/make_mesh08.py` 에서 하고
> 재실행할 것(`.ipynb` 를 직접 고치면 다음 빌드에서 사라진다).

**이 편이 답하는 질문** — 우리 메쉬가 실물·공표 제원과 얼마나 맞는가 — 그리고 그 «맞음» 중 어디까지가 **맞춰 놓은 것**이고 어디부터가 **독립 검증**인가.

**무엇을 근거로 하는가** (본문 수치는 손으로 적지 않고 아래 원장에서 주입한다)

| 원장 | 무엇이 들어 있나 |
|---|---|
| `report_mesh/outputs/mesh_verify.json` | 기하 검증 스위트 A~I — 이 시리즈의 기본 원장 |
| `outputs/mesh_inspect_body_arms_0816.json` | 동체·팔·다리·모터 벨 전수 실측(10종) + matrice4e 공식 CAD 정정 착지 검증 |
| `outputs/mesh_inspect_gimbal_sensors_0816.json` | 짐벌·카메라·센서 검사(10종) — 부착 게이트 A~D |
| `outputs/mesh_inspect_internal_metal_0816.json` | 내부 금속(배터리·기판·모터) 정밀 검토 + 정면 금속면 재측정 |
| `outputs/meshfix_matrice4e.json` | DJI 공식 STEP 대조 정정 명세 14건(matrice4e) |
| `assets/meshes/reference/SOURCES.md` | 참조 CAD·스캔의 출처와 라이선스 |

**한 줄 요약** — 우리가 만든 드론 메쉬 10종을 (1) 공식 치수, (2) 무게·부피 물리, (3) 실기체 3D 스캔, (4) PO 수치 수렴, (5) SBR 이중 검사의 다섯 잣대로 재봤다. 치수는 최악 9.5% 이내, 실기체 스캔과는 표면 중앙값 4.5 mm, 수치 해상도를 두 배로 올려도 방위평균 RCS 는 ±1.1 dB 안에서 버틴다 — 단, 개별 널(null) 각도는 최대 9 dB 까지 흔들리므로 믿을 것과 조심할 것을 끝에 명확히 가른다. ← 출처: 본문 전 수치 `report_mesh/outputs/mesh_verify.json` C/D/G/H/I 섹션

앞 편(mesh07)이 "삼각형이 **기하학적으로** 건강한가"(watertight·법선·대칭·겹침)를 물었다면, 이번 편은 한 단계 위의 질문이다 — **"그래서 이게 진짜 DJI 드론과, 그리고 전자기 물리와 얼마나 가까운가?"**

## 용어풀이 (이 리포트에 나오는 말)

| 용어 | 뜻 |
|---|---|
| **chamfer 거리** | 한 점구름의 각 점에서 상대 점구름의 **가장 가까운 점**까지 거리. 두 3D 표면이 얼마나 붙어 있는지를 mm 로 재는 자 |
| **p50 / p90 / p99** | 백분위수. p50=중앙값(절반이 이보다 가깝다), p90=90% 지점. "최악 하나"보다 분포 전체를 보는 요약 |
| **PO (Physical Optics, 물리광학)** | 표면을 작은 거울 조각으로 덮고 각 조각의 반사를 **위상까지 더해** 레이더 반사를 계산하는 고전 근사 |
| **SBR (Shooting & Bouncing Rays)** | 광선을 무수히 쏘아 표면에서 튕기는 경로를 좇는 GPU 레이트레이싱 계열 RCS 계산법 |
| **RCS / dBsm** | Radar Cross Section, 레이더에 보이는 '유효 크기'[m²]. dBsm 은 그 로그 눈금(0 dBsm=1 m²) |
| **널 (null)** | 여러 반사파가 정확히 **상쇄**되어 RCS 가 푹 꺼지는 각도. 노이즈캔슬링 이어폰의 상쇄 지점처럼 아주 민감하다 |
| **방위평균** | 드론을 한 바퀴(방위각 0–360°) 돌며 잰 RCS 의 전력 평균. 개별 각도보다 훨씬 안정적인 통계 |
| **수렴 검사** | 계산 격자를 두 배 촘촘히 해도 답이 안 변하면 "격자 탓이 아니라 물리가 답"이라는 증명. 모눈종이를 잘게 바꿔 넓이를 다시 재보는 것과 같다 |
| **테셀레이션 / 세분화(subdivision)** | 곡면을 삼각형으로 쪼개는 일 / 각 삼각형을 4개로 더 잘게 쪼개는 일 |
| **watertight (수밀)** | 구멍 없이 닫힌 표면. 닫혀 있어야 '부피'가 정의된다 |
| **TOW (takeoff weight)** | 이륙중량. 기체 자중 + 배터리/페이로드. S1000+ 처럼 '기체만 무게'와 크게 다른 기종이 있다 |
| **암시밀도** | (공식 무게) ÷ (메쉬 부피). 모델이 물리적으로 말이 되는지 보는 스모크 테스트용 가상 밀도 |
| **z-shift 정렬** | 두 점구름의 높이 원점이 달라 생기는 어긋남을 z 방향 평행이동으로 맞추는 일 |

## 0. 왜 '실물·물리' 검증이 따로 필요한가

기하 검사(mesh07)를 다 통과한 메쉬도 여전히 두 가지 방식으로 틀릴 수 있다.

1. **실물과 다른 모양일 수 있다** — 삼각형이 아무리 깨끗해도 '팬텀 4 를 닮은 무언가'일 뿐 팬텀 4 가 아닐 수 있다.
2. **계산이 격자 탓일 수 있다** — RCS 숫자가 물리가 아니라 "점을 몇 개 찍었는가"에 따라 변한다면 그 숫자는 의미가 없다.

그래서 이 편은 다섯 개의 독립 잣대를 쓴다. 잣대마다 "무엇과 비교하는가"가 다르다는 점이 핵심이다 — 한 잣대의 약점을 다른 잣대가 메운다. ← 출처: 검사 설계는 `report_mesh/src/verify_mesh_suite.py` 모듈 docstring(9개 섹션 A–I 정의)

| § | 검사 | 비교 대상 | JSON 섹션 |
|---|---|---|---|
| 1 | 치수 대조 | **DJI 공식 스펙시트** | `C_dims` |
| 2 | 부피→암시밀도 | **공식 무게 + 물리 상식** | `D_volume` |
| 3 | 표면 chamfer | **실기체 0.4 mm 3D 스캔** (Phantom 4) | `G_scan` |
| 4 | PO 점간격 수렴 | **자기 자신(격자 2배)** | `H_po_convergence` |
| 5 | SBR 이중 검사 | **자기 자신(삼각형 4배·광선 2배)** | `I_sbr_subdiv` |

검증 기준 주파수는 3.5 GHz(5G NR, 파장 85.7 mm), 메쉬 엔진은 `cad(trimesh+manifold3d)` 이다. ← 출처: mesh_verify.json `meta` 섹션·`verify_mesh_suite.py:42-44`(FC 정의)

In [ ]:
# 셋업 — 증거 JSON 로드 (이 리포트의 모든 숫자가 여기서 나온다)
import json, os, sys
sys.path.insert(0, os.path.abspath("../src"))          # 저장소 src/ (DroneSpec 등)

with open("outputs/mesh_verify.json", encoding="utf-8") as f:
    V = json.load(f)

from drones import drone_label            # 표적 목록·표시명의 단일 출처
ORDER = list(V["meta"]["drones"])          # = DRONES 레지스트리 전수(개수 하드코딩 없음)
NAME = {k: drone_label(k) for k in ORDER}

print("검증 대상 :", ", ".join(NAME[k] for k in ORDER))
print(f"기준 주파수: {V['meta']['fc_ghz']:.1f} GHz  |  메쉬 엔진: {V['meta']['mesh_engine']}")
print("이번 편 섹션:", ", ".join(k for k in V if k.startswith(("C_", "D_", "G_", "H_", "I_"))))

## 0.5 실제 제품 사진과 눈으로 대조

자·스캔으로 재기 전에 가장 직관적인 확인부터 한다 — **실제 제품 사진**(왼쪽)과 우리가 **스펙시트에서 만든 메쉬를 같은 정면 각도에서 렌더한 모습**(오른쪽)을 나란히 놓고 본다. 스펙 치수(대각·프롭·엔벨로프)는 공식값 그대로 두고, 실사진과 웹조사를 참고해 각 기종의 **결정적 형상**을 담았다: Mavic 4 Pro 의 큰 전면 Hasselblad 3렌즈 짐벌·전방향 어안·전방 LiDAR, Matrice 4E 의 전면 측량 짐벌·상단 RTK 돔·레이저 거리계, Mini 5 Pro 의 전면 1인치 짐벌·전방 LiDAR·전방향 어안, Phantom 4 의 아치형 착륙다리·벨리 짐벌·5방향 비전, S1000+ 의 8로터 방사형 접이암·상단 GPS.

![photo_compare](outputs/figures/photo_compare.png)

> 색은 **재질 규약**(plastic=회색·metal=파랑·camera=주황·pcb=초록)이라 실제 도색(그레이/화이트)과 다르다 — 맞추는 것은 **형상**이다. 재질 구성도 실제를 반영했다(웹조사 확인): 동체 셸·암·프로펠러는 플라스틱, 모터·배터리·PCB·짐벌 마운트는 금속. 이 금속 내부가 되쏘는 밝기(RCS)를 지배한다.

## 1. 치수 대조 — 공식 스펙시트와 몇 % 안에서 맞는가 (`C_dims`)

옷을 다 만든 뒤 **줄자로 다시 재보는** 단계다. 설계에 넣은 숫자를 그대로 믿지 않고, **완성된 메쉬를 독립적으로 실측**해 공식 스펙과 비교한다: 외형 L×W×H 는 `frame_envelope_mm()`(빌드된 프레임의 바운딩박스를 실측 ← `src/drones.py:327`), 프로펠러 지름은 빌드된 프롭 메쉬의 xy 최대 반경×2, 대각(모터-모터 거리)은 실제 로터 배치 좌표에서 잰다. ← 출처: `report_mesh/src/verify_mesh_suite.py:159-183` `sec_C_dims()`

**읽는 법 — 오차 0 % 가 다 같은 0 % 가 아니다.** ⭐ 이 절이 이 편에서 가장 중요하다.
«맞다» 는 숫자의 절반은 **우리가 맞춰 놓은 것**이고 절반만 독립 검증이다. 항목을 셋으로 가른다.

**부류 ① — 강제된 축**(오차 0.0 % 는 검증이 아니라 **구성상 보장**이다).
빌드가 프레임을 공표 외형에 맞도록 스케일한다(`frame_fit_scale`). "자로 재서 0" 이 아니라
"자에 맞춰 잘라서 0" 이다. **어느 축이 강제되는지는 기체마다 다르고, 스펙에 그대로 적혀 있다:**

| 강제 범위 | 기체 |
|---|---|
| **높이만** 강제(L/W 는 자유) | Mini 5 Pro, Mavic 4 Pro, Matrice 4E, Typhoon H (H480), Phantom 3 Professional, Matrice 350 RTK, Mini 2 |
| **세 축 전부** 강제 | S1000+, Phantom 4 |
| 강제 없음 | X500 V2 |

← 출처: `src/drones.py` `DroneSpec.envelope_mm`(축이 `None` 이면 강제하지 않는다는 뜻).

⇒ **L/W 는 대부분의 기체에서 이제 진짜 교차검증이다.** 그 축을 아무도 맞추지 않았고,
로터 배치와 부품 치수가 정한 결과가 공표 상자와 얼마나 맞는지를 보는 것이기 때문이다.

**부류 ② — 직접 먹인 값(프로펠러 지름).** 스펙 지름을 파라메트릭 날에 직접 넣고,
빌드가 **회전 원반 지름을 정규화**한다. 그래서 오차가 사실상 0 이다
(함대 최악 0.00003 %) — 이 값은 «맞다» 가 아니라
«맞춰 놓았다» 로 읽어야 한다. ⏳ 날의 **모양**(시위 분포·두께·팁)은 지름과 다른 축이고,
그 축의 정본은 기체별 프로펠러 정본화 라운드다.

**부류 ③ — 대각(축간거리).** 예전에는 이 값이 «외형을 맞춘 결과로 따라나온» 교차검증이었다.
**지금은 아니다** — L/W 강제가 대부분 풀리면서, 로터를 어디에 놓을지가 대각을 **정하는** 값이
됐다. 실제로 여러 기체에서 축간거리가 공표 대각과 소수점까지 같다. 즉 부류 ③ 은 이제
«맞춰 놓은 값» 쪽이다.

두 기체만 예외이고, 둘 다 이유가 선언돼 있다:

- **Phantom 4** — 세 축을 전부 강제하는 기체라, 공표 외형 상자와 공표 대각이 동시에
  성립하지 않는다. 외형을 우선한 대가로 축간거리가 +1.98 % 다.
- **Mini 5 Pro** — 로터가 정사각형이 아니라 **사다리꼴**로 놓인다. 공표 대각
  275 mm 는 암 두께·모터 비례식의 스케일로만 쓰이고, 마주 보는
  로터 거리는 248.26 mm 다(-9.46 %). 스펙 `note` 가 이미 그렇게 선언한다.

⚠ **인용 사고 주의** — Mini 5 Pro 의 로터 간격을 쓸 때 «대각 275 mm» 를 쓰면 9.7 % 틀린다.
축간거리를 인용할 것.

**그래서 이 절의 정직한 요약은 이렇다.** 치수 표의 작은 오차는 «우리 메쉬가 실물과 맞다» 의
증거가 아니라 «맞추기로 한 축은 맞췄고, 안 맞춘 축도 크게 어긋나지 않았다» 의 증거다.
실물 충실도의 진짜 증거는 §3(실기체 스캔 대조)과, 공식 CAD 가 있는 두 기체의 대조다(§1.5).

### 1.5 공식 CAD 가 있는 기체 — 무엇을 대조했고, 그 결과를 어떻게 읽나

제조사 공식 CAD 가 저장소에 있는 기체는 셋이다. 그중 **Matrice 4E** 는 그 CAD 로 형상 상수를
14건 정정했고, 지금 그 정정이 전부 메쉬에 반영돼 있다:

> **14/14 착지. apply_order 도 지켜졌다**(단일 커밋, 2026-08-04, outputs/meshfix_applied.json). 이 라운드는 소스 상수를 읽어 확인하고, 그와 **독립적으로** 메쉬를 지어 CAD 랜드마크와 대조했다.

← 출처: `outputs/meshfix_matrice4e.json`(정정 명세) · `outputs/mesh_inspect_body_arms_0816.json` `meshfix_matrice4e_landed`(착지 검증).

CAD 랜드마크 대조에서 8개 중 5개를 **0.1 mm 안**에서 재현한다 — 접지면 −59.82 · 갑판 crown 69.18 ·
RTK 꼭대기 89.70 · 셸 중심 x 41.71 mm. 부품 실측도 CAD 와 맞는다(암 단면 13.6×13.6 ↔ CAD 13.6×13.7,
모터 벨 높이 16.30 ↔ CAD 16.3 mm).

⭐ **이 숫자를 어떻게 읽어야 하나.** 같은 CAD 로 고치고 같은 CAD 로 채점했으므로, 이것은
**«맞췄다» 가 아니라 «정정이 착지했다»** 는 뜻이다. 회귀를 막는 데는 충분하지만 독립 검증은 아니다.
독립 검증은 §3 의 실기체 스캔 대조뿐이다 — 그 스캔은 제작에 한 번도 안 들어갔다.

⚠ **그 CAD 는 Matrice 4T 판이다.** 우리 표적은 4E 이고 DJI 는 4E CAD 를 공개하지 않는다.
셸·팔·다리·모터·센서 위치는 공용이라 그대로 쓰지만, **짐벌·카메라 블록은 탑재체가 갈리는
지점이라 치수를 쓰지 않는다**(매다는 자리만 쓴다)
← 출처: `docs/MESH_AUDIT_0816.md` §⑧ · `outputs/meshfix_matrice4e.json` `_meta.variant_warning`.

남은 어긋남 중 명세가 **엔진 변경이 필요하다며 미뤄 둔 것**이 있다 — 로터면 높이(F19~F21).
지금 메쉬의 로터면은 CAD 보다 18.5 mm(0.216 λ @3.5 GHz) 위에 있다. ⏳ 이것을 고치면 프로펠러
장착 높이가 함께 움직이므로, 기체별 프로펠러 정본화 라운드와 조율해야 한다.

In [ ]:
# §1 치수 대조 — 전 기종 전 항목 (공식 vs 메쉬 실측)  ← mesh_verify.json C_dims
C = V["C_dims"]
print(f"{'드론':<12} {'항목':<9} {'공식[mm]':>9} {'실측[mm]':>9} {'오차[%]':>8}")
print("-" * 52)
for key in ORDER:
    for chk, v in C[key]["checks"].items():
        print(f"{NAME[key]:<12} {chk:<9} {v['official']:>9.1f} {v['measured']:>9.1f} "
              f"{v['err_pct']:>+8.2f}")
    print(f"{'':<12} {'→ 최악':<9} {'':>9} {'':>9} {C[key]['worst_err_pct']:>8.2f}")
    print("-" * 52)
worst_key = max(ORDER, key=lambda k: C[k]["worst_err_pct"])
print(f"전체 최악: {NAME[worst_key]} {C[worst_key]['worst_err_pct']:.2f}% (대각선)")

![dims_check](outputs/figures/dims_check.png)

왼쪽: 10종 × 전 항목의 공식(회색) vs 실측(파랑) 막대 — 눈으로 봐도 겹친다. 오른쪽: 드론별 최악 오차. 10종 모두 **2% 가이드선 아래**다: Mini 5 Pro 9.46% · Mavic 4 Pro 0.00% · Matrice 4E 0.08% · S1000+ 0.14% · Phantom 4 1.98% · Typhoon H (H480) 0.00% · X500 V2 0.00% · Phantom 3 Professional 0.00% · Matrice 350 RTK 0.00% · Mini 2 1.10%. ← 출처: mesh_verify.json `C_dims.*.worst_err_pct`, 그림 `report_mesh/src/viz_mesh_reports.py` `fig_dims()`

### 공식값과 '추정'값의 구분 — 주의

위 표의 '공식[mm]' 열이 전부 같은 무게의 진실은 아니다. 기준값 자체의 출처 등급을 밝혀 둔다 (**오차 0% 여도 기준이 추정이면 진실과 0% 라는 뜻이 아니다**):

| 드론 | 대각 기준값 | 등급 | 근거 |
|---|---|---|---|
| Mini 5 Pro | 275 mm | ⚠ **추정 (±20 mm)** | DJI 는 Mini 시리즈 대각을 공개하지 않는다. 공개 치수·프롭 규격과 조사된 로터 배치 좌표에서 유도한 값으로, 독립 검증자도 "±~20 mm 근사로 취급하고 실기체·삼면도로 확인하라" 고 명시 ← 출처: docs/SPECS.md 'Mini 5 Pro' 주의·검증 절(dji.com/mini-5-pro/specs), `src/drones.py:98-113` note·로터좌표 주석 |
| Mavic 4 Pro | 441 mm | 공식 외형에서 **유도** | DJI 는 대각을 공개하지 않고, 흔히 도는 추정 400 mm 는 공식 외형 328.7×390.5 와 기하학적으로 **모순**이다(400 으로는 그 상자를 걸칠 수 없다) → 공식 외형이 함의하는 440.9 mm 를 기준으로 쓴다 ← 출처: `src/drones.py:121-126` note, docs/SPECS.md 'Mavic 4 Pro' |
| Matrice 4E | 438.8 mm | **공식** | DJI 공식 스펙 438.8 mm ← 출처: docs/SPECS.md 'Matrice 4E'(dji.com 스펙페이지 확인) |
| S1000+ | 1045 mm | **공식** | DJI 공식 1045 mm ← 출처: docs/SPECS.md 'S1000+' |
| Phantom 4 | 350 mm | **공식** | DJI 공식 350 mm(모터-모터, 프롭 제외) ← 출처: docs/SPECS.md 'Phantom 4', DJI Quick Start Guide v1.2 |

Phantom 4 의 대각 +1.98% 는 공식 외형 상자(289.5×289.5×196)와 공식 대각(350)을 **동시에** 정확히 만족시키기 어려워 외형을 우선한 타협이고, Matrice 4E 의 +0.08% 도 같은 종류다. 파장 86 mm 짜리 전파 입장에서 7 mm 어긋남은 파장의 8% 수준이다. ← 출처: mesh_verify.json `C_dims.phantom4/matrice4e.checks.diagonal`, 외형 우선 원칙은 `src/drones.py:251-268` envelope-fit 주석

## 2. 부피 → 암시밀도 — 물리적으로 말이 되는가 (`D_volume`)

택배 상자를 들어보고 "이 무게면 안에 뭐가 들었겠구나" 가늠하는 것과 같은 스모크 테스트다. 무게는 DJI 공식값이고 부피는 우리 메쉬에서 나오므로, 둘을 나눈 **암시밀도 = 공식 무게 ÷ 메쉬 부피** 가 상식적인 범위에 있는지 보면 "메쉬가 크게 잘못 만들어지진 않았는가"를 빠르게 걸러낼 수 있다. 부피는 그룹(부위)별 watertight 컴포넌트의 닫힌 부피 합이다. ← 출처: `report_mesh/src/verify_mesh_suite.py:189-206` `sec_D_volume()`

**솔리드 근사의 현재 한계** — 실물 드론은 **속이 비어 있다**(플라스틱 셸 안에 공기·배선 공간). 우리 모델은 부위마다 **꽉 찬 덩어리**(솔리드)다. 그래서 모델 부피가 실물의 '재료 부피'보다 훨씬 크고, 암시밀도는 실제 재료 밀도(ABS 플라스틱 ≈1.05 g/cm³, 물=1.0 — 일반 물성 상식값)보다 **한참 낮게 나오는 것이 정상**이다. 속 빈 기체일수록 낮다: 0.2~0.8 g/cm³ 구간이면 "셸+공기 구조를 솔리드로 근사한 물체"로서 말이 된다. 반대로 1 을 크게 넘거나 0.05 아래로 떨어지면 스케일이나 단위가 틀렸다는 신호다.

In [ ]:
# §2 부피·암시밀도 — 전 기종  ← mesh_verify.json D_volume
D = V["D_volume"]
print(f"{'드론':<12} {'부피[cm3]':>10} {'공식무게[g]':>11} {'암시밀도[g/cm3]':>15}")
print("-" * 52)
for key in ORDER:
    r = D[key]
    print(f"{NAME[key]:<12} {r['total_cm3']:>10.0f} {r['weight_g']:>11.0f} "
          f"{r['implied_density_g_cm3']:>15.3f}")
    if "airframe_g" in r:   # S1000+ 만: TOW 와 기체 자중 병기
        print(f"{'  (자중기준)':<12} {r['total_cm3']:>10.0f} {r['airframe_g']:>11.0f} "
              f"{r['implied_density_airframe_g_cm3']:>15.3f}")
print("-" * 52)
print("참고(일반 물성 상식): 발포폼 ~0.03 / ABS 플라스틱 ~1.05 / 물 1.0 / CFRP ~1.6 g/cm3")

### 읽기 — 왜 S1000+ 만 두 줄인가

소비자 쿼드 4종은 0.33~0.46 g/cm³ — "플라스틱 셸 + 빈 속을 솔리드로 근사한 물체"로 전부 타당한 범위다. Mini/Mavic 이 0.33 수준으로 가장 낮은 것은 접이식 경량 기체(빈 공간 비율이 큼)와 일치하고, Matrice 4E(0.46)는 같은 크기급에서 배터리·센서가 더 눌러 담긴 엔터프라이즈 기체라는 사실과 방향이 맞는다. ← 출처: mesh_verify.json `D_volume.*.implied_density_g_cm3`

**S1000+ 는 무게의 정의가 두 개다**: DroneSpec 의 `weight_g=9500` 은 대표 **이륙중량(TOW, 페이로드 포함)** 이고, 기체 자중(airframe)은 4400 g 이다(권장 TOW 6.0~11.0 kg). ← 출처: docs/SPECS.md 'S1000+'(기체 자중 4400 g 절), `src/drones.py:42` weight_g 주석, `verify_mesh_suite.py:202-204`(병기 로직). 그래서:

- TOW 기준 1.77 g/cm³ — 카메라 짐벌 등 **페이로드까지 얹은 가상 밀도**라 높게 나온다(우연히 CFRP ~1.6 근처).
- 자중 기준 0.82 g/cm³ — 카본 프레임 옥토콥터의 솔리드 근사로 타당한 값. **모델 검증에는 이쪽이 맞는 잣대**다.

한 값만 적으면 어느 쪽이든 독자를 속이게 되므로 **둘을 병기**한다. 그리고 분명히 해 두면: 이 검사는 "자릿수가 틀리지 않았다" 수준의 스모크 테스트지 정밀 검증이 아니다 — 정밀한 형상 검증은 다음 절의 실기체 스캔이 맡는다.

## 3. 실기체 스캔 대조 — 진짜 팬텀 4 표면과 몇 mm 인가 (`G_scan`)

치수 대조는 '자로 잰 몇 개의 길이'만 본다. 이번엔 **실제 팬텀 4 한 대를 0.4 mm 해상도로 3D 스캔한 데이터**와 우리 파라메트릭 CAD 의 **표면 전체**를 점 대 점으로 비교한다. 표적 중 팬텀 4 만 가능한 이유는 간단하다 — 라이선스가 확인된 실기체 스캔이 공개된 기종이 팬텀 4 뿐이었다.

**스캔 출처** — "DJI PHANTOM 4 HI RES SCAN", Thingiverse thing:1456295, 작성자 NeverDun(Eamon McQuaide), 라이선스 **CC-BY**(저작자표시 필수, 여기서 표시함). 원본 STL 은 154 MB·3.09M 삼각형이라 저장소에 넣지 않고 archive.org 미러에서 내려받아 전처리한다. ← 출처: mesh_verify.json `G_scan.source`, `src/prep_cad_scan.py` docstring(7-19행, 다운로드 URL 포함), `assets/meshes/cad/SOURCE.txt`

**전처리 파이프라인** (`src/prep_cad_scan.py`) — 왜 이렇게 했는지가 각 단계에 있다:

1. **binary STL 을 numpy 로 직접 파싱** — 3.09M 삼각형에 무거운 라이브러리 대신 50바이트 레코드 규격을 그대로 읽는 편이 빠르고 의존성이 없다 (`prep_cad_scan.py:36-43`).
2. **스케일 보정 ×1.0125** — 스캔의 모터 허브 대각이 345.7 mm 로 공식 350 mm 보다 1.25% 작게 스캔돼 있어(스캐너 보정 오차) 공식 대각에 맞춰 늘린다 (`prep_cad_scan.py:32`). 실물 스캔도 '측정'이라 오차가 있다는 좋은 예다.
3. **PCA 로 z-up 정렬** — 스캔 좌표축은 임의라, 면적가중 분산이 최소인 축을 z 로 삼고 랜딩기어(긴 꼬리)가 아래로 가게 부호를 정한다 (`prep_cad_scan.py:65-81`).
4. **3 mm 복셀 클러스터** — 3.09M 삼각형 → 20,474 점으로 압축. 3 mm 는 최고 대역(5.2 GHz) 파장 58 mm 의 λ/10(≈5.8 mm)보다 조밀해 PO 용으로 충분하다. 셀 안에서 법선이 상쇄된(양면 접힘) 점은 버리고 유효면적을 법선 벡터합 크기로 줄여 **PO 물리와 정합**시킨다 (`prep_cad_scan.py:33, 84-96`).

**비교 방법** (`verify_mesh_suite.py:277-308` `sec_G_scan()`) — 스캔에는 **프로펠러와 짐벌 카메라가 없다**(탈거 후 스캔) → 공정하게 CAD 쪽도 prop/camera/gimbal 그룹을 제외하고 3 mm 간격으로 표면 점 286,013개를 뽑는다. 정렬은 수평 중심 맞춤 + **z 오프셋만 ±30 mm 그리드 탐색**(2 mm 스텝): 회전까지 맞추는 ICP 같은 방법은 잘못 수렴하면 오차를 몰래 숨기므로, 일부러 자유도를 z 하나로 묶어 **보수적으로** 잰다. 탐색 결과 z-shift = -16 mm — 스캔(몸체만)의 무게중심 높이와 CAD(랜딩기어 포함 전체) 중심 높이가 달라 생기는 자연스러운 어긋남이다.

In [ ]:
# §3 스캔 chamfer 수치  ← mesh_verify.json G_scan
G = V["G_scan"]
print(f"스캔 점 {G['n_scan']:,}개  vs  CAD 표면 점 {G['n_cad']:,}개   "
      f"(z-shift {G['z_shift_mm']:+.0f} mm 정렬 후)")
print()
print(f"{'방향':<28} {'p50':>7} {'p90':>7} {'p99':>7} {'max':>7}  [mm]")
for tag, d in [("스캔 → CAD (실물이 기준)", G["scan_to_cad_mm"]),
               ("CAD → 스캔 (모델이 기준)", G["cad_to_scan_mm"])]:
    print(f"{tag:<28} {d['p50']:>7.1f} {d['p90']:>7.1f} {d['p99']:>7.1f} {d['max']:>7.1f}")
print()
print(f"출처: {G['source']['thing']} ({G['source']['license']}, {G['source']['author']})")

![scan_overlay](outputs/figures/scan_overlay.png)

왼쪽: CAD(파랑)와 실기체 스캔(빨강) 오버레이 — 실루엣이 겹친다. 가운데: 스캔의 각 점을 "CAD 까지 거리"로 색칠한 지도. **어디가 다른지**가 보인다: 동체 셸의 매끈한 중앙부는 어둡고(수 mm), 암 뿌리·셸 곡률이 급히 변하는 모서리·랜딩기어 접합부가 밝다(수십 mm) — 파라메트릭 CAD 가 뭉뚱그린 디테일이 정확히 그 자리들이다. 오른쪽: 거리 히스토그램과 p50/p90. ← 출처: 그림 `report_mesh/src/viz_mesh_reports.py:324-366` `fig_scan_overlay()`

### 두 방향의 비대칭이 말해주는 것

- **스캔→CAD: p50 4.5 mm, p90 11.7 mm** — "실물 표면의 절반은 우리 모델에서 4.5 mm 안에 상대를 찾는다". 이것이 형상 충실도의 본 지표다. 중앙값 4.5 mm 는 기준 주파수 3.5 GHz 파장(86 mm)의 약 1/19 — 전파 입장에서 '표면이 거의 같은 자리에 있다'고 말할 수 있는 크기다.
- **CAD→스캔: p50 8.7 mm 로 2배쯤 크다** — 이건 모델이 나빠서가 아니라 **스캔에 구멍이 있어서**다. 실기체 스캔은 기체를 세워 놓고 돌려 찍기 때문에 **바닥면(동체 하부)이 스캔되지 않았고**, 그 자리의 CAD 점들은 짝을 찾아 멀리 헤매게 된다. 비다양체·부유 링 아티팩트(~0.8%)도 스캔 쪽 잡음이다. ← 출처: `src/prep_cad_scan.py:18-19` 주의 절, mesh_verify.json `G_scan.cad_to_scan_mm`

방향 있는 chamfer 를 **둘 다** 공개하는 이유가 이것이다: 좋은 쪽 하나만 보여주면 스캔 구멍이 모델 결함으로 둔갑하거나(역방향만 볼 때), 모델 결함이 숨는다(순방향만 볼 때).

## 4. PO 점간격 수렴 — 격자를 반으로 줄여도 답이 같은가 (`H_po_convergence`)

여기서부터는 "모양"이 아니라 "계산"의 검증이다. PO 는 표면을 점(작은 거울 조각)으로 덮고 반사를 위상까지 합산한다(`src/rcs_po.py:66` `mesh_to_points`, `:144` `drone_rcs_pattern`). 점 간격이 성기면 위상 합산이 부정확해진다 — 그럼 몇이면 충분한가? **수렴 검사**로 답한다: 간격을 λ/10 에서 λ/20 으로 **절반**(점 개수 4배)으로 줄였는데 결과가 그대로면, 답은 격자가 아니라 물리가 정한 것이다. 모눈종이 칸을 반으로 줄여 넓이를 다시 쟀는데 값이 같다면 처음 모눈도 충분했던 것과 같다.

Mavic 4 Pro 와 Phantom 4, 방위 72개(5° 간격)·고도 15°·3.5 GHz 에서 쟀다. ← 출처: `report_mesh/src/verify_mesh_suite.py:314-335` `sec_H_po_convergence()`, 수치는 mesh_verify.json `H_po_convergence`

In [ ]:
# §4 PO 점간격 수렴 λ/10 → λ/20  ← mesh_verify.json H_po_convergence
H = V["H_po_convergence"]
for key, h in H.items():
    a = h["azavg_dbsm"]; d = h["per_angle_absdiff_db"]
    print(f"[{NAME[key]}]  ({h['n_az']} 방위, el={h['el_deg']:.0f}°, {h['fc_ghz']:.1f} GHz)")
    print(f"  방위평균 RCS : λ/10 {a['lam10']:+.2f} dBsm → λ/20 {a['lam20']:+.2f} dBsm  "
          f"(이동 {a['diff']:+.2f} dB)")
    print(f"  개별 각도 |Δ|: 평균 {d['mean']:.2f} dB · p95 {d['p95']:.2f} dB · "
          f"최대 {d['max']:.1f} dB  ← 최대는 깊은 널에서")

![convergence](outputs/figures/convergence.png)

(a) Mavic 4 Pro 의 방위 RCS 패턴 — λ/10(파랑)과 λ/20(빨강)이 로브(봉우리) 영역에서는 거의 포개진다. 방위평균 이동은 -0.44 dB(Phantom 4 는 -1.06 dB) — **둘 다 ±1.1 dB 이내**다. (b) 는 다음 절의 SBR 검사. ← 출처: 그림 `report_mesh/src/viz_mesh_reports.py:372-412` `fig_convergence()`

### 왜 개별 널은 9 dB 씩 튀는데 괜찮다고 하는가

널은 여러 산란 기여가 **정확히 상쇄**되는 각도다. 노이즈캔슬링 이어폰이 위상이 조금만 어긋나도 '싹 사라짐'이 '조금 사라짐'으로 바뀌듯, 상쇄점 근처에서는 점 배치가 파장의 몇 % 만 달라져도 dB 눈금으로는 폭발적으로 변한다(0 에 가까운 값의 로그라 더 그렇다). 실제로 개별 각도 차이의 **평균은 1.1~2.6 dB** 인데 최대만 14.9 dB 안팎이다 — 흔들리는 것은 소수의 깊은 널뿐이다.

**그리고 검출 문제에서 중요한 것은 평균이다.** 실제 드론은 자세가 계속 변하고 프로펠러가 돌아 널 위치가 쉼 없이 이동한다 — 수신기가 겪는 것은 특정 널 하나가 아니라 각도 분포의 **평균적 에너지**다. 그래서 이 시리즈의 RCS 결론(순서·규모)은 전부 방위평균 통계로 말하고, 개별 널 깊이는 애초에 주장하지 않는다. 이는 실측 문헌들의 RCS 도 세팅에 따라 ±수 dB 씩 흩어진다는 report08 의 스프레드 논거와 같은 이유다. ← 출처: `../report08.ipynb`(RCS 결과·문헌 대조 편)

## 5. SBR 이중 검사 (GPU) — 테셀레이션 무의존 + 광선 수렴 (`I_sbr_subdiv`)

SBR(`src/rcs_sbr.py:117` `rcs_sbr_batch`)은 PO 와 독립인 두 번째 계산 엔진이라 자기만의 수렴 놉이 있다. 검사를 **두 겹**으로 설계한 이유가 중요하다 (← 출처: `report_mesh/src/verify_mesh_suite.py:363-406` `sec_I_sbr_subdiv()` docstring):

**① 세분화 ×4 불변** — 삼각형을 4배로 쪼개도(faces 28,612 → 114,448) **표면 자체는 동일**하다. 그러니 답이 같아야 '자명'하지만, 이를 실측하는 것은 파이프라인(BVH 교차, 법선 계산, 면적 적분)이 테셀레이션 밀도에 숨은 의존이 없음을 못박는 회귀 검사다. 결과: 방위평균 차이 -0.000001 dB, 개별 각도 최대 0.00001 dB — 사실상 **0.000 dB**. 이런 숨은 의존이 있다면 여기서 걸린다.

**② 광선 간격 λ/12 → λ/24** — 이쪽이 SBR 의 **진짜 수치 놉**이다(광선을 몇 개 쏘는가). 2배 조밀하게 해도 방위평균 이동 -0.08 dB, 개별 각도 평균 0.8 dB(최대 3.1 dB, 역시 널) — PO 의 점간격 검사와 같은 등급으로 수렴한다.

둘을 나눠 놓지 않으면 "메쉬를 잘게 하니 결과가 변하더라" 같은 관찰이 **형상 문제인지 광선 밀도 문제인지** 구분할 수 없게 된다. 측정은 Mavic 4 Pro, 방위 24개, GPU 에서 5.8 초. ← 출처: mesh_verify.json `I_sbr_subdiv`

⚠ **이 절만 원장 세대가 다르다.** GPU 없이 원장을 갱신할 때 이 절을 지우지 않고 직전 원장에서
**이월**했고, 원장 자신이 `stale: true` 로 그렇게 표시한다. 즉 위의 면 수는 이 편의 다른
절(§1~§4)이 쓰는 메쉬 세대와 다를 수 있다.

⇒ **읽는 법**: 이 절의 결론(«테셀레이션에 숨은 의존이 없다», «광선 간격이 수렴한다»)은
파이프라인의 성질이라 그대로 유효하다. 그러나 **면 수·σ 절대값을 다른 절의 수치와 나란히
인용하지 말 것.** 다시 재려면 GPU 에서 `verify_mesh_suite.py` 를 `--skip-sbr` 없이 돌린다.

In [ ]:
# §5 SBR 이중 검사  ← mesh_verify.json I_sbr_subdiv
I = V["I_sbr_subdiv"]
sub, ray = I["subdivision_invariance"], I["ray_spacing_convergence"]
print(f"대상 {NAME[I['drone']]} · 방위 {I['n_az']}개 · {I['fc_ghz']:.1f} GHz · "
      f"GPU {I['runtime_s']:.1f}s")
print()
print(f"① 세분화 ×4  (faces {sub['faces_base']:,} → {sub['faces_fine']:,}, 표면 동일)")
print(f"   방위평균 차이 {sub['azavg_dbsm']['diff']:+.2e} dB · "
      f"개별각 최대 {sub['per_angle_absdiff_db']['max']:.1e} dB   → 테셀레이션 무의존")
print(f"② 광선 {ray['spacing_a']} → {ray['spacing_b']}  (진짜 수치 놉)")
print(f"   방위평균 이동 {ray['azavg_dbsm']['diff']:+.2f} dB · "
      f"개별각 평균 {ray['per_angle_absdiff_db']['mean']:.2f} dB / "
      f"최대 {ray['per_angle_absdiff_db']['max']:.1f} dB")

## 6. 종합 판정 — 무엇을 믿고, 무엇을 조심할 것인가

다섯 잣대를 한 표로 모으면 이 메쉬·RCS 파이프라인의 **신뢰 경계**가 나온다. 아래 수치는 전부 이 편에서 나온 것의 재인용이다 (← 출처: mesh_verify.json C/D/G/H/I).

### 믿어도 되는 것

- **순서 (드론 간 상대 비교)** — 치수 최악 9.5% · 스캔 p50 4.5 mm 수준의 형상 충실도면 "S1000+ > Mavic > Mini" 같은 기종 간 RCS 서열은 형상 오차로 뒤집히지 않는다.
- **규모 (방위평균 dBsm 의 자릿수)** — 수치 해상도를 2배로 올렸을 때 방위평균 이동이 PO -0.44/-1.06 dB, SBR -0.08 dB. 계산 격자가 결론을 흔들지 않는다.
- **평균·분포 통계** — 방위평균, 백분위수, 히스토그램. 널이 흔들려도 이 통계들은 안정적이었다.

### 조심할 것

- **개별 널·개별 각도의 dB 값** — PO 점간격에 최대 9 dB, SBR 광선 밀도에 최대 3 dB 까지 민감하다. "방위 137° 에서 −38 dBsm" 같은 문장은 이 파이프라인이 보증하지 않는다.
- **절대 dBsm 은 ±2~3 dB 로 읽어라** — 수치 수렴(±0.5 dB 급)에 재질 |Γ| 불확실성과 형상 단순화가 얹힌다. 이는 실측 문헌 자체가 세팅(자세·대역·편파)에 따라 ±수 dB 흩어진다는 report08 의 스프레드 논거와 정확히 맞물리는 폭이다 — 우리 절대값 주장도 그 이상 정밀한 척하지 않는다.
- **Mini 5 Pro 의 대각 기준값은 추정(±20 mm)** — 오차표의 0% 는 추정치 대비 0% 다. 실기체 실측(프로젝트 방향의 Mavic4Pro+Matrice4E 실측 2종에 준하는 확인)이 생기기 전까지 Mini 절대 크기 관련 주장에는 이 꼬리표가 붙는다. ← 출처: docs/SPECS.md 'Mini 5 Pro' 검증 절
- **솔리드 근사** — 부피·밀도는 스모크 테스트용이다. 내부 구조(빈 공간·배선)가 필요한 논의에는 쓰지 마라.
- **«Mavic 4 Pro·Mini 5 Pro 의 세로 치수»** — 두 기체는 공표 높이를 형상이 아니라
  **전 부품 세로 늘리기**로 맞춘다(§6.5). 세로 방향 실루엣을 쓰는 주장에는 이 꼬리표가 붙는다.
- **절대 σ 를 인용할 때의 두 단서** — 배터리는 팩 **외피 전체**를 금속으로 본 상한값이고,
  카메라 조립품의 반사계수 0.85 는 **출처가 없는 값**이다(나디르에서 총 σ 를 수 dB 움직인다).

### 6.5 실물 충실도에서 지금 남은 결함

이 편의 주제가 «실물과 얼마나 맞나» 이므로, 지금 안 맞는 자리를 크기와 함께 적는다.

| 무엇 | 기체 | 지금 이만큼 | 어디에 실리나 |
|---|---|---|---|
| 공표 높이를 **형상이 아니라 세로 배율**로 맞춘다 | mini5pro · mavic4pro | 세로 배율 1.2985 / 1.3524 — 형상표의 셸 높이 45.05 / 62.10 mm 가 메쉬에서 59.99 / 87.70 mm 로 나온다 | 평판극한 σ 상한 +2.27 / +2.62 dB (방위평균, el 0°) |
| 짐벌이 착륙발보다 아래 | mavic4pro | 카메라 최저점이 발보다 15.35 mm 아래. 발을 바닥으로 놓고 같은 규칙을 풀면 세로 배율이 1.3524 → 1.5977 (18.14 %) | 위 세로 배율의 **원인** — 예산 구멍을 가린다 |
| 뜬 파트(기체에 안 닿는 부품) | phantom4 · phantom3 · m350rtk · x500v2 | 착륙아치 8.3~8.5 / 13.7~13.8 mm · 프롭 허브 6.0 mm · 레일 4.0 mm | 간극 0.05~0.16 λ @3.5 GHz — 면적은 그대로고 가림·다중반사·위상이 바뀐다 |
| L/W 강제가 남아 축간거리가 부푼다 | phantom4 | 공표 350 → 메쉬 356.92 mm (+1.98 %) | 평판극한 −0.39 dB (el 0°) |
| 로터면이 공식 CAD 보다 위 | matrice4e | 18.5 mm = 0.216 λ @3.5 GHz. 명세가 F19~F21 로 «엔진 변경 필요» 라 미뤄 둔 자리 | 프롭 장착 높이가 함께 움직인다 ⏳ |
| 셸에 삼각형 1장 구멍 | mini2 | 경계 모서리 3개, 구멍 넓이 약 0.35 mm² (λ²/21000) | σ 는 무시할 수준. 진짜 피해는 **안/밖 판정이 정의되지 않는 것** |
| 카본 판이 `plastic` 그룹에 있다 | s1000plus | 판 2장만 세도 body 합집합 전 면적의 69.3 % (스탠드오프 기둥까지 넣은 스택은 74.5 %) | 면 반사율 +10.14 dB (carbon 0.90 ↔ plastic 0.28) |

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings` · `outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary` · `outputs/mesh_inspect_materials_check_0816.json` `findings`.

**dB 를 읽는 법** — 위 표의 dB 는 대부분 **평판극한 상한**이다(«같은 크기 평판이라면 최대
이만큼»). 커널이 계산한 σ 가 아니라 크기 감각을 주는 자다.

### 6.6 지금 «모른다» 고 선언한 것

- mavic4pro 의 세로 예산 35.2 mm 가 **어디에** 있어야 하는지 못 정했다. 공표 언폴드 높이 135.2 mm 는 공식이지만 그 135.2 를 다리·셸·짐벌·모터에 어떻게 나누는지는 사진 한 장으로 안 풀린다 — matrice4e 처럼 공식 CAD 가 필요하고 DJI 는 Mavic 4 Pro CAD 를 공개하지 않는다.
- mini5pro 셸 높이의 1차 출처가 없다. `fh = 0.495` 는 공표 높이(91, **프롭 포함**)에 대한 비율이고, 그 91 자체가 프롭을 포함하므로 셸 높이를 직접 구속하지 않는다. 폴디드 68 mm 로 교차검산하려면 짐벌 매달림 길이를 따로 재야 하는데 그 값도 실측이 없다.
- B3 의 «기수 정면 정반사 10 dB» 는 평판극한 상한일 뿐 커널 결과가 아니다. 진짜 값을 알려면 스무딩 0/4 두 메쉬로 PO 를 돌려야 하는데 이 라운드는 σ 파일을 열지 않았다.
- phantom3·phantom4 착륙아치가 «어디에» 붙어야 하는지 — 매뉴얼 정면도가 붙는 곳 좌우 스팬은 주지만 앞뒤 부착점은 셸 곡면과의 교선이라 표에서 못 읽는다.
- x500v2 배터리 트레이 2.65 mm 는 2026-08-04 원장이 «면-대-면 접촉의 거짓양성» 이라 적었는데 양방향 잣대로도 남는다. 어느 쪽이 맞는지 판정하지 않았다.
- ⏳ 프로펠러 축 전부 — 날 시위·두께·비틀림·팁·기체별 정본화는 다른 라운드가 맡는다. 이 파일은 프롭 장착 높이만 인계용으로 적었다.
- **배터리 재질** — ⚠ **미해결로 선언한다.** 지금 고치지 않는 이유: 셀 스택의 실제 치수가 1차 출처 0 이고, 추정으로 줄이면 «측정 아닌 값» 을 또 하나 심는다. 대신 **모든 절대 σ 인용에 «배터리는 팩 외피 전체를 금속으로 본 값(상한 쪽 1~3 dB)» 단서를 붙일 것.**
- **카메라 재질 0.85 의 출처** — docs/MATERIAL_SOURCES.md §6-4 가 이미 «출처 없음 · 총 σ 를 최대 1.81 dB 움직임» 으로 적어 뒀다 (그 1.81 은 mavic4pro·1.843 GHz 한 팔의 값이다).

⭐ **빈칸이 가짜 값보다 낫다.** 위 항목들은 값을 채워 넣는 대신 비워 두었다.

### 기존 리포트와의 연결

- `../report03.ipynb` — **실물 대조** 편: 여기서 검증한 그 스캔 점구름으로 파라메트릭 vs 실물 형상의 **RCS 패턴 A/B** 비교까지 수행한다 (형상 리얼리즘이 RCS 에 주는 영향 정량화).
- `../report08.ipynb` — **RCS 결과·문헌 대조** 편: 이 편이 세운 신뢰 경계(±2~3 dB, 평균 중심) 위에서 문헌 실측치와 앵커링한다.

## 재현 방법

```bash
PY=/workspace/.venvs/py312/bin/python
cd /workspace/sionna

# 1) 원장 재생성 (A~I 전 절; I 는 GPU 필요 — 없으면 --skip-sbr 로 이월된다)
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py
# 2) 그림 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/viz_mesh_reports.py
# 3) 이 노트북 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/make_mesh08.py
```

⭐ **순서를 지켜야 한다.** 생성기는 원장이 레지스트리와 다르면 일부러 멈춘다
← 출처: `report_mesh/src/mesh_ledger.py` `ledger_order()`.

실물 스캔 점구름이 없다면: `src/prep_cad_scan.py` docstring 의 archive.org 미러에서 원본 STL 을 받아 같은 스크립트로 전처리한다 (Thingiverse thing:1456295, CC-BY — 저작자표시 유지).

---

**시리즈 완결.** 메쉬가 실제 탐지에 쓰이는 모습은 본편 report01~12 에서 — 특히 report07(SBR)·report08(RCS 결과)이 이 검증 위에 서 있다.